# TALLER 3: Feature Engineering - Creando Variables Poderosas

## Descripcion
Feature Engineering es el proceso de crear variables nuevas y relevantes a partir de los datos existentes. Es la tecnica que mas impacto tiene en la calidad de un modelo de Machine Learning, mas incluso que elegir el mejor algoritmo.

## Por que importa?
Los modelos de ML solo saben hacer aritmetica — nosotros les damos las variables correctas para aprender.

Un modelo es como un estudiante que solo sabe aritmetica basica. No puede 'leer' una fecha y entender que diciembre = temporada alta de ventas. Pero si tu le das la variable 'es_diciembre = 1', ahora si puede detectar ese patron.

## Los 5 tipos de transformaciones:
1. **Extraccion** — Sacar informacion oculta dentro de una variable compuesta
2. **Combinacion** — Crear nuevas variables combinando dos o mas columnas
3. **Transformacion matematica** — Aplicar funciones para normalizar la distribucion
4. **Agregacion historica** — Calcular estadisticas del comportamiento pasado
5. **Encoding y Escalado** — Convertir variables categoricas a numericas y alinear rangos

In [ ]:
# Codigo de partida para el Taller 3
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Cargar los datos (usamos el dataset limpio del Taller 2)
df = pd.read_csv('ventas.csv', parse_dates=['fecha'])

# Aplicar limpieza basica
df_clean = df.copy()
df_clean = df_clean.drop_duplicates(subset=['pedido_id'], keep='first')
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'])
for col in ['precio', 'monto', 'costo', 'items']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean['ciudad'] = df_clean['ciudad'].fillna(df_clean['ciudad'].mode()[0])
df_clean['items'] = df_clean['items'].fillna(df_clean['items'].median())
df_clean['costo'] = df_clean['costo'].fillna(df_clean['costo'].median())

print('Dataset listo para Feature Engineering!')
print(f'Shape: {df_clean.shape}')
print(f'Columnas: {list(df_clean.columns)}')

## Ejercicio 1: Extraccion de caracteristicas de fecha

**Objetivo:** Extraer componentes utiles de la columna fecha.

**Explicación:**
Una columna de fecha contiene mucha informacion oculta: mes, dia de la semana, trimestre, si es fin de semana, etc.

**Justificacion:**
- El mes puede indicar temporada alta/baja
- El dia de la semana puede mostrar patrones de compra
- Los trimestres son utiles para analisis financiero

In [ ]:
# Ejercicio 1: Extraccion de caracteristicas de fecha
print('=' * 60)
print('EJERCICIO 1: EXTRACCION DE FECHA')
print('=' * 60)

# Asegurar que fecha es datetime
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'])

# Extraer componentes de la fecha
df_clean['año'] = df_clean['fecha'].dt.year
df_clean['mes'] = df_clean['fecha'].dt.month
df_clean['dia'] = df_clean['fecha'].dt.day
df_clean['dia_semana'] = df_clean['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_clean['trimestre'] = df_clean['fecha'].dt.quarter
df_clean['semana_del_año'] = df_clean['fecha'].dt.isocalendar().week

# Crear variables binarias
df_clean['es_finde_semana'] = (df_clean['dia_semana'] >= 5).astype(int)
df_clean['es_mes_fin'] = df_clean['dia'].apply(lambda x: 1 if x >= 25 else 0)
df_clean['es_diciembre'] = (df_clean['mes'] == 12).astype(int)

# Nombre del dia de la semana
dias = {0: 'Lunes', 1: 'Martes', 2: 'Miercoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sabado', 6: 'Domingo'}
df_clean['nombre_dia'] = df_clean['dia_semana'].map(dias)

print('Nuevas columnas creadas:')
print(['año', 'mes', 'dia', 'dia_semana', 'trimestre', 'semana_del_año', 
       'es_finde_semana', 'es_mes_fin', 'es_diciembre', 'nombre_dia'])
print()
print(df_clean[['fecha', 'año', 'mes', 'dia_semana', 'trimestre', 'es_finde_semana', 'es_diciembre']].head(10))

## Ejercicio 2: Combinacion de columnas

**Objetivo:** Crear nuevas variables combinando columnas existentes.

**Explicación:**
A veces la relacion entre dos variables es mas util que las variables por separado.

**Justificacion:**
- El margen de ganancia es mas util que solo el monto
- El precio promedio por item revela patrones de compra
- La razon costo/venta indica eficiencia

In [ ]:
# Ejercicio 2: Combinacion de columnas
print('=' * 60)
print('EJERCICIO 2: COMBINACION DE COLUMNAS')
print('=' * 60)

# Margen de ganancia (utilidad)
df_clean['margen'] = df_clean['monto'] - df_clean['costo']
df_clean['margen_porcentaje'] = (df_clean['margen'] / df_clean['monto']) * 100

# Precio promedio por item
df_clean['precio_promedio_item'] = df_clean['monto'] / df_clean['items']

# Razon costo/venta
df_clean['razon_costo_venta'] = df_clean['costo'] / df_clean['monto']

# Ticket promedio (para este dataset, usamos el monto como ticket)
# Esta tam es una medida deano de compra
df_clean['tamano_compra'] = pd.cut(df_clean['monto'], 
                                     bins=[0, 1000, 5000, 10000, float('inf')],
                                     labels=['pequeño', 'mediano', 'grande', 'premium'])

print('Nuevas columnas creadas:')
print(['margen', 'margen_porcentaje', 'precio_promedio_item', 'razon_costo_venta', 'tamano_compra'])
print()
print(df_clean[['monto', 'costo', 'margen', 'margen_porcentaje', 'precio_promedio_item']].head(10))

## Ejercicio 3: Transformacion matematica

**Objetivo:** Aplicar transformaciones para normalizar distribuciones y reducir skewness.

**Explicación:**
Los algoritmos de ML asumen que los datos siguen una distribucion normal. Cuando los datos tienen skewness (asimetria), podemos transformarlos.

**Justificacion:**
- Log transform reduce el efecto de valores extremos
- Square root puede ayudar con datos positivamente sesgados
- Many ML algorithms perform better with normalized distributions

In [ ]:
# Ejercicio 3: Transformacion matematica
print('=' * 60)
print('EJERCICIO 3: TRANSFORMACION MATEMATICA')
print('=' * 60)

# Ver skewness actual
print('Skewness antes de transformacion:')
print(f'  monto: {df_clean["monto"].skew():.2f}')
print(f'  precio: {df_clean["precio"].skew():.2f}')
print()

# Transformacion log (solo para valores positivos)
# Agregamos 1 para evitar log(0)
df_clean['log_monto'] = np.log1p(df_clean['monto'])
df_clean['log_precio'] = np.log1p(df_clean['precio'])

# Transformacion sqrt
df_clean['sqrt_monto'] = np.sqrt(df_clean['monto'])

# Transformacion Box-Cox (mas robusta)
from scipy import stats

# Solo aplicar a valores positivos y no nulos
monto_positivo = df_clean['monto'].dropna()
monto_positivo = monto_positivo[monto_positivo > 0]
df_clean['boxcox_monto'] = df_clean['monto'].apply(
    lambda x: stats.boxcox(x + 1)[0] if x > 0 else np.nan
)

print('Skewness despues de transformacion:')
print(f'  log_monto: {df_clean["log_monto"].skew():.2f}')
print(f'  sqrt_monto: {df_clean["sqrt_monto"].skew():.2f}')

## Ejercicio 4: Agregacion historica

**Objetivo:** Calcular estadisticas del comportamiento pasado de cada entidad.

**Explicación:**
Agregamos informacion historica por cliente, vendedor o producto para capturar patrones de comportamiento.

**Justificacion:**
- Un cliente con historial de compras altas es mas probable que compre de nuevo
- Un vendedor con buenas ventas historicas seguira rindiendo bien
- Permite al modelo aprender de patrones temporales

In [ ]:
# Ejercicio 4: Agregacion historica por vendedor
print('=' * 60)
print('EJERCICIO 4: AGREGACION HISTORICA')
print('=' * 60)

# Agregacion por vendedor
agregacion_vendedor = df_clean.groupby('vendedor_id').agg({
    'monto': ['sum', 'mean', 'count', 'std'],
    'pedido_id': 'count',
    'categoria': 'nunique'
}).reset_index()

# Aplanar nombres de columnas
agregacion_vendedor.columns = ['vendedor_id', 'ventas_totales_vendedor', 
                               'venta_promedio_vendedor', 'num_pedidos_vendedor',
                               'desvest_ventas_vendedor', 'num_pedidos_vendedor2',
                               'categorias_vendedor']

# Eliminar columna duplicada
agregacion_vendedor = agregacion_vendedor.drop('num_pedidos_vendedor2', axis=1)

# Llenar NaN en desviacion estandar con 0
agregacion_vendedor['desvest_ventas_vendedor'] = agregacion_vendedor['desvest_ventas_vendedor'].fillna(0)

# Merge con el dataset original
df_clean = df_clean.merge(agregacion_vendedor, on='vendedor_id', how='left')

print('Agregaciones por vendedor creadas:')
print(agregacion_vendedor.head(10))

In [ ]:
# Ejercicio 4: Agregacion por region
print('=' * 60)
print('AGREGACION POR REGION')
print('=' * 60)

# Agregacion por region
agregacion_region = df_clean.groupby('region').agg({
    'monto': ['sum', 'mean', 'count']
}).reset_index()

agregacion_region.columns = ['region', 'ventas_totales_region', 
                             'venta_promedio_region', 'num_pedidos_region']

# Merge
df_clean = df_clean.merge(agregacion_region, on='region', how='left')

print(agregacion_region)

## Ejercicio 5: Encoding de variables categoricas

**Objetivo:** Convertir variables categoricas a numericas para que los algoritmos puedan procesarlas.

**Explicación:**
- **One-Hot Encoding:** Crea columnas binarias para cada categoria (pd.get_dummies)
- **Label Encoding:** Asigna un numero a cada categoria
- **Ordinal Encoding:** Para categorias con orden natural

**Importante:** Usar drop_first=True en regresion para evitar multicolinealidad.

In [ ]:
# Ejercicio 5: Encoding de variables categoricas
print('=' * 60)
print('EJERCICIO 5: ENCODING DE VARIABLES CATEGORICAS')
print('=' * 60)

# Ver categorias unicas
print('Categorias unicas:')
print(f'  Region: {df_clean["region"].nunique()} valores - {list(df_clean["region"].unique())}')
print(f'  Categoria: {df_clean["categoria"].nunique()} valores - {list(df_clean["categoria"].unique())}')
print(f'  Ciudad: {df_clean["ciudad"].nunique()} valores')
print()

# One-Hot Encoding para region y categoria
df_encoded = pd.get_dummies(df_clean, columns=['region', 'categoria'], drop_first=True)

# Label Encoding para ciudad (alto cardinalidad)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_encoded['ciudad_encoded'] = le.fit_transform(df_encoded['ciudad'].astype(str))

print('Columnas despues de encoding:')
nuevas_columnas = [col for col in df_encoded.columns if col.startswith('region_') or col.startswith('categoria_')]
print(nuevas_columnas)

## Ejercicio 6: Escalado de variables

**Objetivo:** Alinear los rangos de las variables para que los algoritmos funcionen correctamente.

**Explicación:**
Las variables con rangos muy distintos hacen que los algoritmos ignoren las de menor escala.

- **StandardScaler:** Media = 0, Desv. = 1 (para datos normales)
- **MinMaxScaler:** Rango [0, 1] (para redes neuronales, KNN)

**REGLA DE ORO - Data Leakage:**
Ajusta (fit) el scaler SOLO en X_train → luego transforma X_train y X_test por separado.

In [ ]:
# Ejercicio 6: Escalado de variables
print('=' * 60)
print('EJERCICIO 6: ESCALADO DE VARIABLES')
print('=' * 60)

# Seleccionar columnas numericas para escalar
columnas_a_escalar = ['monto', 'precio', 'items', 'costo', 'margen', 
                      'ventas_totales_vendedor', 'venta_promedio_vendedor']

# Verificar que existan
columnas_existentes = [col for col in columnas_a_escalar if col in df_encoded.columns]
print(f'Columnas a escalar: {columnas_existentes}')
print()

# Ejemplo: StandardScaler
scaler = StandardScaler()
df_encoded[['monto_scaled', 'precio_scaled', 'items_scaled']] = scaler.fit_transform(
    df_encoded[['monto', 'precio', 'items']]
)

print('StandardScaler - Media y Desv Estandar despues del escalado:')
print(f'  monto_scaled: media={df_encoded["monto_scaled"].mean():.4f}, std={df_encoded["monto_scaled"].std():.4f}')
print(f'  precio_scaled: media={df_encoded["precio_scaled"].mean():.4f}, std={df_encoded["precio_scaled"].std():.4f}')
print()

# Ejemplo: MinMaxScaler
minmax = MinMaxScaler()
df_encoded[['monto_minmax', 'precio_minmax', 'items_minmax']] = minmax.fit_transform(
    df_encoded[['monto', 'precio', 'items']]
)

print('MinMaxScaler - Min y Max despues del escalado:')
print(f'  monto_minmax: min={df_encoded["monto_minmax"].min():.4f}, max={df_encoded["monto_minmax"].max():.4f}')
print(f'  precio_minmax: min={df_encoded["precio_minmax"].min():.4f}, max={df_encoded["precio_minmax"].max():.4f}')

## Ejercicio 7: Resumen de features creadas

**Objetivo:** Ver un resumen de todas las features creadas y su justificacion.

In [ ]:
# Ejercicio 7: Resumen de features creadas
print('=' * 60)
print('RESUMEN DE FEATURES CREADAS')
print('=' * 60)

# Listar todas las columnas nuevas
columnas_originales = ['pedido_id', 'fecha', 'region', 'ciudad', 'categoria', 
                        'items', 'precio', 'monto', 'costo', 'vendedor_id']

nuevas_features = [col for col in df_encoded.columns if col not in columnas_originales]

print(f'Total de columnas originales: {len(columnas_originales)}')
print(f'Total de features nuevas: {len(nuevas_features)}')
print()
print('Features creadas:')
for i, col in enumerate(nuevas_features, 1):
    print(f'  {i}. {col}')

In [ ]:
# Ejercicio 7: Tabla de features con justificacion
print('=' * 60)
print('FEATURES CREADAS CON JUSTIFICACION')
print('=' * 60)

features_justificacion = [
    ('año', 'Extraccion', 'El ano puede capturar tendencias anuales'),
    ('mes', 'Extraccion', 'El mes indica temporada alta/baja'),
    ('trimestre', 'Extraccion', 'Util para analisis financiero'),
    ('es_finde_semana', 'Extraccion', 'Patrones de compra en fin de semana'),
    ('es_diciembre', 'Extraccion', 'Temporada alta de ventas'),
    ('margen', 'Combinacion', 'Utilidad directa de cada venta'),
    ('margen_porcentaje', 'Combinacion', 'Porcentaje de ganancia'),
    ('precio_promedio_item', 'Combinacion', 'Valor promedio por item'),
    ('log_monto', 'Transformacion', 'Reduce skewness, normaliza distribucion'),
    ('ventas_totales_vendedor', 'Agregacion', 'Historial de rendimiento del vendedor'),
    ('venta_promedio_vendedor', 'Agregacion', 'Consistencia del vendedor'),
    ('ventas_totales_region', 'Agregacion', 'Tamano del mercado regional'),
    ('region_* (one-hot)', 'Encoding', 'Convierte categorias a binarias'),
    ('categoria_* (one-hot)', 'Encoding', 'Convierte categorias a binarias'),
    ('monto_scaled', 'Escalado', 'Estandariza para algoritmos sensibles a escala'),
    ('monto_minmax', 'Escalado', 'Normaliza a rango 0-1'),
]

df_features = pd.DataFrame(features_justificacion, columns=['Feature', 'Tipo', 'Justificacion'])
print(df_features.to_string(index=False))

## Ejercicio 8: Pipeline completo de Feature Engineering

**Objetivo:** Crear un pipeline que ejecute todos los pasos de manera ordenada.

In [ ]:
# Ejercicio 8: Pipeline completo
print('=' * 60)
print('PIPELINE COMPLETO DE FEATURE ENGINEERING')
print('=' * 60)

# Paso 1: Cargar y limpiar
df_final = pd.read_csv('ventas.csv', parse_dates=['fecha'])
df_final = df_final.drop_duplicates(subset=['pedido_id'], keep='first')
df_final['fecha'] = pd.to_datetime(df_final['fecha'])
for col in ['precio', 'monto', 'costo', 'items']:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
df_final['ciudad'] = df_final['ciudad'].fillna(df_final['ciudad'].mode()[0])
df_final['items'] = df_final['items'].fillna(df_final['items'].median())
df_final['costo'] = df_final['costo'].fillna(df_final['costo'].median())

print(f'Paso 1 - Carga y limpieza: {df_final.shape}')

# Paso 2: Extraccion
df_final['año'] = df_final['fecha'].dt.year
df_final['mes'] = df_final['fecha'].dt.month
df_final['trimestre'] = df_final['fecha'].dt.quarter
df_final['dia_semana'] = df_final['fecha'].dt.dayofweek
df_final['es_finde_semana'] = (df_final['dia_semana'] >= 5).astype(int)
df_final['es_diciembre'] = (df_final['mes'] == 12).astype(int)

print(f'Paso 2 - Extraccion: {df_final.shape}')

# Paso 3: Combinacion
df_final['margen'] = df_final['monto'] - df_final['costo']
df_final['margen_porcentaje'] = (df_final['margen'] / df_final['monto']) * 100
df_final['precio_promedio_item'] = df_final['monto'] / df_final['items']

print(f'Paso 3 - Combinacion: {df_final.shape}')

# Paso 4: Agregacion
agg_vendedor = df_final.groupby('vendedor_id').agg({
    'monto': ['sum', 'mean', 'count']
}).reset_index()
agg_vendedor.columns = ['vendedor_id', 'ventas_totales_vendedor', 
                       'venta_promedio_vendedor', 'num_pedidos_vendedor']
df_final = df_final.merge(agg_vendedor, on='vendedor_id', how='left')

print(f'Paso 4 - Agregacion: {df_final.shape}')

# Paso 5: Encoding
df_final = pd.get_dummies(df_final, columns=['region', 'categoria'], drop_first=True)

print(f'Paso 5 - Encoding: {df_final.shape}')

# Paso 6: Escalado
scaler = StandardScaler()
cols_to_scale = ['monto', 'precio', 'items', 'margen']
df_final[[f'{c}_scaled' for c in cols_to_scale]] = scaler.fit_transform(df_final[cols_to_scale])

print(f'Paso 6 - Escalado: {df_final.shape}')

print()
print('=' * 60)
print(f'DATASET FINAL: {df_final.shape[0]} filas, {df_final.shape[1]} columnas')

## Resumen del Taller 3

En este taller cubrimos las tecnicas fundamentales de Feature Engineering:

1. **Extraccion:** Sacar informacion oculta de fechas y texto
2. **Combinacion:** Crear ratios y metricas derivadas
3. **Transformacion:** Normalizar distribuciones con log, sqrt, etc.
4. **Agregacion:** Calcular estadisticas historicas por entidad
5. **Encoding:** Convertir categoricas a numericas
6. **Escalado:** Estandarizar rangos de variables

**Nota importante:** El Feature Engineering puede mejorar el rendimiento de un modelo 30-50%, mucho mas que cambiar el algoritmo (5-10%).